In [1]:
import re
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.nn as nn

In [2]:
df = pd.read_csv('/kaggle/input/datasets/bitext/bitext-gen-ai-chatbot-customer-support-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')
df.head()
df.shape
df.columns
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:


def check_placeholders(text):
    # List of specific words or regex patterns to look for
    patterns = [
        r"YOUR_[A-Za-z_]+",  # Catches YOUR_DATASET, YOUR_FILE, etc.
        r"\bunknown\b",      # Catches the exact word "unknown"
        r"todo",             # Catches "todo"
        r"<.*?>",            # Catches anything in angle brackets like <insert path>
    ]
    
    found = []
    for pattern in patterns:
        # re.IGNORECASE makes it catch "UNKNOWN", "Unknown", etc.
        matches = re.findall(pattern, str(text), flags=re.IGNORECASE) 
        if matches:
            found.extend(matches)
            
    if found:
        print(f"⚠️ Placeholder alert! Found: {set(found)}")
    else:
        print("✅ All clear! No placeholders detected.")
check_placeholders(df)

✅ All clear! No placeholders detected.


In [4]:
df["intent"].value_counts()

intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [5]:
df["instruction"].str.len().describe()
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [6]:
X = df['instruction']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)


In [7]:
text = X_train.tolist()
def tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

tokens = [tokenize(sentence) for sentence in text]

In [8]:
vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for token_list in tokens:
    for word in token_list:
        if word not in vocab:
            vocab[word] = len(vocab)

def encode(token_list):
    return [vocab.get(word, vocab['<UNK>']) for word in token_list]

encoded_text = [encode(token_list) for token_list in tokens]
# print(f'{tokens[0]} : {encoded_text[0]}') 


In [9]:
max_length = max(len(x) for x in encoded_text)

padded_text = []
for token_list in encoded_text:
    padded_length = max_length - len(token_list)
    token_list += [0] * (padded_length)
    padded_text.append(token_list)
# padded_text

In [10]:
labels = sorted(y_train.unique())
labels_to_id ={
    label:i
    for i, label in enumerate(labels)
}

id_to_labels ={
    i:label
    for label, i in labels_to_id.items()
}

encoded_labels = [labels_to_id[label] for label in y_train]


In [11]:
X_train_tensor = torch.tensor(padded_text, dtype=torch.long)
y_train_tensor = torch.tensor(encoded_labels, dtype=torch.long)

In [12]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)


In [13]:
class ChatbotNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_length, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.network = nn.Sequential(
            nn.Linear(max_length * embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        return self.network(x)

In [14]:
vocab_size = len(vocab)
embedding_dim = 64
num_classes = len(labels_to_id)

model = ChatbotNN(
    vocab_size,
    embedding_dim,
    max_length,
    num_classes
)

print(model)

ChatbotNN(
  (embedding): Embedding(2558, 64, padding_idx=0)
  (network): Sequential(
    (0): Linear(in_features=1088, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=27, bias=True)
  )
)


In [15]:

X_batch, y_batch = next(iter(train_loader))
output = model(X_batch)


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 0.01)
loss = criterion(output, y_batch)


In [17]:
epochs = 120
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0
    for batch_x, batch_y in train_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predictions = output.argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
    accuracy = correct/total
    avg_loss = total_loss/ len(train_loader)
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Accuracy: {accuracy:.4f}"
    )

Epoch [1/120] Loss: 0.0159 Accuracy: 0.9971
Epoch [2/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [3/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [4/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [5/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [6/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [7/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [8/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [9/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [10/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [11/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [12/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [13/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [14/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [15/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [16/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [17/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [18/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [19/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [20/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [21/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [22/120] Loss: 0.0000 Accuracy: 1.0000
Epoch [23/120] Loss

In [18]:
model.eval()
correct = 0
with torch.no_grad():
    for X_batch, y_batch in train_loader:
        output = model(X_batch)

        predictions = output.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 9.24%


In [19]:


# torch.save({
#     "model_state_dict": model.state_dict(),
#     "vocab": vocab,
#     "intent_to_idx": intent_to_idx,
#     "idx_to_intent": idx_to_intent,
#     "embedding_dim": 64,
#     "num_classes": len(intent_to_idx)
# }, "/kaggle/working/chatbot_model.pth")

# print("Model saved!")

In [20]:
print(output.shape)
print(output[0])
print(y_batch[0])

torch.Size([57, 27])
tensor([-12.6966,   0.0840, -23.8812,  -5.6386,   4.6804, -14.2210, -25.2165,
        -10.9626, -25.7485,  -9.6231,  -7.2890, -13.5449, -22.6388, -12.4254,
        -11.1816,   2.6432, -13.7770, -15.0873, -15.0221, -27.5788, -17.8049,
          7.1668, -41.8454, -16.3924, -18.4420, -12.8497, -40.3930])
tensor(7)


In [21]:
from IPython.display import Audio, display

display(Audio(
    data=[0, 1] * 1000,
    rate=2000,
    autoplay=True
))